# Résoudre `2_medium` avec OR-Tools
Les notebooks 09 et 10 construisaient une solution **pas à pas** : règles gloutonnes, puis recuit simulé. Ici, on change d'approche. On **écrit le problème en équations**, et un solveur cherche la meilleure solution de ces équations, avec une **preuve** de l'écart à l'optimum.

C'est la démarche la plus proche d'un physicien : modéliser, puis résoudre exactement. La difficulté est ailleurs. Le modèle ne peut pas contenir les 10 000 datasets un par un. Il raisonne sur des **quantités agrégées**, et il faudra ensuite traduire sa réponse en datasets réels.

**Prérequis** : `datasets/2_medium.json`, `test_solution.py`, `solutions/2_medium_plan_e.json` (créé par le notebook 10), `pandas`, `ortools` (`uv add ortools`).

**Durée d'exécution** : environ 6 minutes, car on laisse le solveur travailler 2 minutes à chaque résolution.

## 1. Charger l'instance et les repères

In [1]:
from pathlib import Path
from collections import defaultdict
import sys, json, time, bisect, os
import pandas as pd
from ortools.linear_solver import pywraplp
from ortools.sat.python import cp_model
pd.set_option("display.width", 150)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from test_solution import getSolutionScore

DATASET = ROOT_DIR / "datasets" / "2_medium.json"
DATASET_TXT = DATASET.read_text()
instance = json.loads(DATASET_TXT)
energy_cap = instance["energyCap"]

VALUE  = {m["id"]: m["value"] for m in instance["models"]}
ENERGY = {m["id"]: m["energyCost"] for m in instance["models"]}
NEEDS  = {m["id"]: [(t, b["lowerBound"], b["upperBound"]) for t, b in m["dataRequirements"].items()] for m in instance["models"]}
SIZE   = {d["id"]: d["size"] for d in instance["datasets"]}
TYPE   = {d["id"]: d["dataType"] for d in instance["datasets"]}
COPY   = {d["id"]: d["isCopyrighted"] for d in instance["datasets"]}
IMPOSSIBLE = {m for m, needs in NEEDS.items() if any(lo > hi for _, lo, hi in needs)}
LOWER = {m: {t: lo for t, lo, hi in needs if lo <= hi} for m, needs in NEEDS.items()}   # besoins remplissables
MONO = {m: needs[0][0] for m, needs in NEEDS.items() if len(needs) == 1 and m not in IMPOSSIBLE}   # sources possibles
FREE_SUPPLY  = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t and COPY[d] == 0) for t in "ntic"}
TOTAL_SUPPLY = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t) for t in "ntic"}

WINNER = 272_322                                  # meilleur score du concours sur 2_medium
plan_e = json.loads((ROOT_DIR / "solutions" / "2_medium_plan_e.json").read_text())
score_e = getSolutionScore(json.dumps(plan_e), DATASET_TXT, show_details=False)[0]
print(f"plan E (notebook 09) : {score_e:,} | plan F (notebook 10) : 271,874 | gagnant : {WINNER:,}")
print(f"modèles impossibles : {len(IMPOSSIBLE)} | sources possibles (modèles à un seul besoin) : {len(MONO)}")

plan E (notebook 09) : 271,511 | plan F (notebook 10) : 271,874 | gagnant : 272,322
modèles impossibles : 58 | sources possibles (modèles à un seul besoin) : 1241


## 2. Correction : le gagnant dépasse ma borne du notebook 10
Le notebook 10 avait calculé une borne de **272 301** en supposant que les modèles impossibles ne rapportaient rien, faute de sources rentables. Le gagnant a fait **272 322**. L'hypothèse était donc fausse : il a forcément tiré des points des modèles impossibles, en les nourrissant avec des sources.

Mon essai du notebook 09 ne gagnait que 4 points avec les sources, parce que la stratégie gloutonne était mauvaise, pas parce que les sources sont inutiles. Le modèle ci-dessous **inclut les sources**, et c'est le solveur qui décidera lesquelles valent le coup.

## 3. Le modèle mathématique
**Variables** (toutes binaires, 0 ou 1) :
- $x_i$ : le modèle $i$ est entraîné **à pleine valeur**, donc uniquement avec des données libres ;
- $y_i$ : le modèle $i$ est entraîné **à demi-valeur**, avec n'importe quelles données ;
- $z_m$ : le modèle impossible $m$ est entraîné (forcément à demi-valeur, puisqu'il utilise des sources) ;
- $u_{s,m,t}$ : le modèle $s$ (à un seul besoin, de type $t$) sert de **source** au besoin impossible $t$ du modèle $m$.

**Objectif** : $\max \; \sum_i v_i\,x_i + \lfloor v_i/2\rfloor\,y_i \;+\; \sum_m \lfloor v_m/2\rfloor\,z_m$

**Contraintes** :
1. un modèle a un seul rôle : $x_i + y_i + \sum_{m,t} u_{i,m,t} \le 1$ (pleine valeur, demi-valeur ou source) ;
2. chaque besoin impossible d'un modèle entraîné a exactement une source : $\sum_s u_{s,m,t} = z_m$ ;
3. énergie : $\sum_i e_i (x_i + y_i) + \sum_m e_m z_m + \sum e_s\,u_{s,m,t} \le E_{\max}$. Une source coûte son énergie ;
4. données libres, par type $t$ : $\sum_i L_{i,t}\,x_i \le \text{libre}_t$ ;
5. données totales, par type $t$ : tous les modèles entraînés et toutes les sources tiennent dans l'offre totale.

$L_{i,t}$ est le besoin minimal du modèle $i$ en type $t$. C'est **l'agrégation** : on ne dit pas quels datasets, seulement combien de données. Cela rend le modèle soluble, mais optimiste. Toute solution réelle respecte ces contraintes, donc l'optimum du modèle est une **borne supérieure** du vrai problème. On ne prévoit des sources que pour les besoins impossibles : pour un besoin remplissable, une source coûterait de l'énergie sans rien rapporter, puisque les données totales ne manquent pas.

In [2]:
def solve_model(solver="SCIP", seconds=120, forbid_full=(), free_margin=1.0):
    """Construit et résout le modèle agrégé. forbid_full : modèles interdits à pleine valeur."""
    t0 = time.time()
    if solver == "SCIP":
        S = pywraplp.Solver.CreateSolver("SCIP")
        new_bool, add = (lambda name: S.BoolVar(name)), S.Add
    else:
        M = cp_model.CpModel()
        new_bool, add = M.new_bool_var, M.add

    x = {m: new_bool(f"x{m}") for m in NEEDS if m not in IMPOSSIBLE}
    y = {m: new_bool(f"y{m}") for m in NEEDS if m not in IMPOSSIBLE}
    z = {m: new_bool(f"z{m}") for m in IMPOSSIBLE}
    u = {(s, m, t): new_bool(f"u{s}_{m}_{t}")
         for m in IMPOSSIBLE for t, lo, hi in NEEDS[m] if lo > hi
         for s, ts in MONO.items() if ts == t}

    as_source = defaultdict(list)
    for (s, m, t), var in u.items():
        as_source[s].append(var)
    sources_of = defaultdict(list)
    for (s, m, t), var in u.items():
        sources_of[(m, t)].append(var)

    for m in x:                                                        # 1. un seul rôle
        add(x[m] + y[m] + sum(as_source[m]) <= 1)
    for m in forbid_full:
        add(x[m] == 0)
    for (m, t), vars_ in sources_of.items():                           # 2. une source par besoin impossible
        add(sum(vars_) == z[m])
    add(sum(ENERGY[m] * (x[m] + y[m]) for m in x) + sum(ENERGY[m] * z[m] for m in z)    # 3. énergie
        + sum(ENERGY[s] * var for (s, m, t), var in u.items()) <= energy_cap)
    for t in "ntic":
        add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m]) <= int(FREE_SUPPLY[t] * free_margin))   # 4. libres
        add(sum(LOWER[m][t] * (x[m] + y[m]) for m in x if t in LOWER[m])                                  # 5. totales
            + sum(LOWER[m][t] * z[m] for m in z if t in LOWER[m])
            + sum(LOWER[s][t] * var for (s, m, tt), var in u.items() if tt == t) <= TOTAL_SUPPLY[t])
    objective = sum(VALUE[m] * x[m] + (VALUE[m] // 2) * y[m] for m in x) + sum((VALUE[m] // 2) * z[m] for m in z)

    if solver == "SCIP":
        S.Maximize(objective); S.SetTimeLimit(seconds * 1000); S.Solve()
        val, obj, bound = (lambda v: round(v.solution_value())), S.Objective().Value(), S.Objective().BestBound()
    else:
        M.maximize(objective)
        cp = cp_model.CpSolver(); cp.parameters.max_time_in_seconds = seconds; cp.parameters.num_workers = os.cpu_count()
        cp.solve(M)
        val, obj, bound = cp.value, cp.objective_value, cp.best_objective_bound
    sol = dict(full=[m for m in x if val(x[m])], half=[m for m in x if val(y[m])],
               impossible=[m for m in z if val(z[m])], sources=[(s, m, t) for (s, m, t), var in u.items() if val(var)])
    info = dict(solveur=solver, objectif=round(obj), borne=round(bound), écart_prouvé=round(bound - obj),
                secondes=round(time.time() - t0), pleins=len(sol["full"]), demi=len(sol["half"]),
                impossibles=len(sol["impossible"]), sources=len(sol["sources"]))
    return sol, info

print("variables de source u :", sum(1 for m in IMPOSSIBLE for t, lo, hi in NEEDS[m] if lo > hi for s in MONO if MONO[s] == t))

variables de source u : 25069


## 4. Résoudre : SCIP contre CP-SAT
OR-Tools propose plusieurs solveurs. On en compare deux :
- **SCIP** : un solveur de **programmation linéaire en nombres entiers** (MIP). Il résout la relaxation linéaire (celle du notebook 09), puis la découpe par *branch and bound* jusqu'à obtenir des 0 et des 1. Il est très bon quand la relaxation est presque entière, comme ici ;
- **CP-SAT** : le solveur de **programmation par contraintes** de Google. Il combine plusieurs stratégies en parallèle (propagation logique, recherche locale, relaxation linéaire).

Les deux donnent un **objectif** (la meilleure solution trouvée) et une **borne** (on a prouvé qu'on ne peut pas faire mieux). Si on les laissait tourner assez longtemps, les deux se rejoindraient et on aurait l'optimum prouvé du modèle.

In [3]:
print("cœurs disponibles :", os.cpu_count())
sol_scip, info_scip = solve_model("SCIP", seconds=120)
sol_cpsat, info_cpsat = solve_model("CP-SAT", seconds=60)
solver_table = pd.DataFrame([info_scip, info_cpsat]).set_index("solveur")
solver_table

cœurs disponibles : 2


,objectif,borne,écart_prouvé,secondes,pleins,demi,impossibles,sources
solveur,,,,,,,,
SCIP,272307,272396,89,121,536,329,8,9
CP-SAT,239090,287009,47919,60,321,408,10,11


**Lecture**
- La **borne** du solveur est d'environ 272 390. Aucune solution ne peut faire mieux, à condition qu'aucune source ne serve à un besoin remplissable (hypothèse de la partie 3). Le gagnant (272 322) est en dessous : tout est cohérent.
- L'**objectif** de SCIP dépasse 272 300 : sur le papier, le modèle fait presque aussi bien que le gagnant. Il entraîne quelques modèles impossibles grâce à des sources.
- CP-SAT est très loin (environ 230 000 à 240 000 après 60 s), et sa borne reste très lâche (287 009). Ce n'est pas une question de cœurs : il fait **moins bien avec 4 cœurs qu'avec 2**. La raison tient à la **structure du problème** :
  - les coefficients sont énormes (énergies jusqu'à $10^{12}$, plafond de $2 	imes 10^{14}$). CP-SAT raisonne d'abord par propagation logique, ce qui marche mal sur une grande contrainte de type sac à dos avec de tels coefficients ;
  - SCIP s'appuie sur la **relaxation linéaire**, et on a vu dans le notebook 09 qu'elle est presque entière (7 modèles fractionnaires sur 3 000). SCIP part donc quasiment de la bonne réponse.
- Deux réglages aideraient CP-SAT : `cp.parameters.linearization_level = 2` (utiliser davantage la relaxation linéaire) et `M.add_hint(...)` (partir de la solution de SCIP).

### Les sources choisies par le solveur

In [4]:
rows = []
for s, m, t in sol_scip["sources"]:
    rows.append(dict(source=s, cible=m, type=t, valeur_source=VALUE[s], énergie_source=ENERGY[s],
                     valeur_cible=VALUE[m], gain_cible=VALUE[m] // 2, énergie_cible=ENERGY[m]))
display(pd.DataFrame(rows))
gain = sum(VALUE[m] // 2 for m in sol_scip["impossible"])
print(f"valeur apportée par les modèles impossibles : {gain}")

,source,cible,type,valeur_source,énergie_source,valeur_cible,gain_cible,énergie_cible
0,1523,2958,n,23,5433845679,57,28,1
1,2783,2958,t,23,1,57,28,1
2,1430,2846,n,9,1,67,33,1
3,501,595,t,21,1,51,25,1
4,1549,2645,i,5,1,8,4,1
5,94,1883,n,9,1,55,27,1
6,2721,860,c,27,1,55,27,1
7,882,1265,i,1,1,92,46,56382447030
8,1529,891,c,29,1,87,43,28704473523


valeur apportée par les modèles impossibles : 233


**Lecture**
- Presque toutes les sources et presque toutes les cibles ont une énergie **quasi nulle** (1). Une source de faible valeur (souvent moins de 30) est sacrifiée pour une cible qui rapporte un peu plus à demi-valeur.
- Le gain total est modeste, environ 230 points, mais à ce niveau de précision il est décisif : c'est lui qui permet de dépasser la borne du notebook 10.
- Le solveur fait l'appariement **globalement** : il considère toutes les combinaisons source-cible en même temps. L'essai glouton du notebook 09, qui traitait les cibles une par une, n'en trouvait qu'une seule.

## 5. Du modèle agrégé aux datasets
Le solveur a choisi **quels modèles** entraîner et à quelle valeur. Il faut maintenant leur attribuer des **datasets réels**. On reprend `Pool` et `fill` des notebooks précédents, avec une amélioration de `fill` : s'il n'existe pas un dataset qui termine le remplissage, on cherche une **paire** de datasets qui le termine (méthode des deux pointeurs), avant de prendre le plus gros qui rentre. Cela gaspille moins les dernières données libres.

In [5]:
class Pool:
    """Réserve de datasets d'un type et d'un statut, triée par taille."""
    def __init__(self, ids):
        self.ids = sorted(ids, key=SIZE.get)
        self.sizes = [SIZE[d] for d in self.ids]
    def take(self, i):
        self.sizes.pop(i)
        return self.ids.pop(i)
    def put(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        self.sizes.insert(i, SIZE[d]); self.ids.insert(i, d)
    def remove(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        while self.ids[i] != d:
            i += 1
        self.take(i)

def find_pair(sizes, a, b):
    """Deux positions i < j avec sizes[i] + sizes[j] dans [a, b], somme la plus petite. Deux pointeurs : O(n)."""
    i, j, best = 0, bisect.bisect_right(sizes, b) - 1, None
    while i < j:
        s = sizes[i] + sizes[j]
        if s > b:
            j -= 1
        elif s < a:
            i += 1
        else:
            if best is None or s < best[0]:
                best = (s, i, j)
            j -= 1
    return best

def fill(pool, lower, upper):
    """Prend des datasets dont la somme tombe dans [lower, upper]. Rend la liste, ou None (rien n'est pris)."""
    taken, total = [], 0
    while total < lower:
        a, b = lower - total, upper - total
        i = bisect.bisect_left(pool.sizes, a)
        if i < len(pool.sizes) and pool.sizes[i] <= b:          # 1. un dataset termine
            total += pool.sizes[i]; taken.append(pool.take(i)); continue
        pair = find_pair(pool.sizes, a, b) if pool.sizes and a <= 2 * pool.sizes[-1] else None
        if pair:                                                 # 2. une paire termine
            _, i, j = pair
            total += pool.sizes[i] + pool.sizes[j]
            taken.append(pool.take(j)); taken.append(pool.take(i)); continue
        j = bisect.bisect_right(pool.sizes, b) - 1               # 3. le plus gros qui rentre
        if j < 0:
            break
        total += pool.sizes[j]; taken.append(pool.take(j))
    if lower <= total <= upper:
        return taken
    for d in taken:
        pool.put(d)
    return None

def new_pools():
    return {(t, c): Pool([d for d in SIZE if TYPE[d] == t and COPY[d] == c]) for t in "ntic" for c in (0, 1)}

def give(pools, model, needs, statuses):
    """Remplit les besoins donnés. Rend la liste [dataset, modèle], ou None (tout est rendu)."""
    got = []
    for t, lo, hi in needs:
        res = None
        for s in statuses:
            res = fill(pools[(t, s)], lo, hi)
            if res is not None:
                break
        if res is None:
            for d in got:
                pools[(TYPE[d], COPY[d])].put(d)
            return None
        got += res
    return [[int(d), int(model)] for d in got]

def pack(sol):
    """Attribue les datasets à la sélection du solveur. Rend (plan, modèles pleins ratés)."""
    pools, mappings, failed = new_pools(), [], []
    # Les pleins d'abord, des plus petits besoins aux plus grands : les petits besoins se casent partout,
    # les grands profitent de ce qui reste. (L'ordre inverse rate beaucoup plus de modèles.)
    for m in sorted(sol["full"], key=lambda m: sum(LOWER[m].values())):
        got = give(pools, m, NEEDS[m], (0,))
        if got is None:
            failed.append(m)
        else:
            mappings += got
    for m in sol["half"] + failed:                               # demi-valeur : sous droits d'abord
        mappings += give(pools, m, NEEDS[m], (1, 0)) or []
    for s, m, t in sol["sources"]:                               # sources : leur valeur ne compte pas
        mappings += give(pools, s, NEEDS[s], (1, 0)) or []
    for m in sol["impossible"]:                                  # cibles : seulement leurs besoins remplissables
        mappings += give(pools, m, [n for n in NEEDS[m] if n[1] <= n[2]], (1, 0)) or []
    plan = {"dataMappings": mappings, "modelMappings": [[int(s), int(m)] for s, m, t in sol["sources"]]}
    return plan, failed

results = {}
def evaluate(name, plan):
    score, valid, message = getSolutionScore(json.dumps(plan), DATASET_TXT, show_details=False)
    results[name] = dict(score=score, valide=valid, écart_au_gagnant=score - WINNER, part_du_gagnant=f"{score / WINNER:.3%}")
    print(f"{name} : score = {score:,} | valide = {valid} {message} | écart au gagnant = {score - WINNER:+,}")
    return score

plan_g, failed = pack(sol_scip)
print("modèles pleins sans données libres suffisantes :", failed, "| perte :", sum(VALUE[m] - VALUE[m] // 2 for m in failed))
evaluate("G : solveur + datasets", plan_g)

modèles pleins sans données libres suffisantes : [1770, 2146] | perte : 992
G : solveur + datasets : score = 271,315 | valide = True  | écart au gagnant = -1,007


271315

**Lecture** : c'est le prix de l'agrégation. Le solveur utilise 99,8 à 99,97 % des données libres de chaque type, en supposant qu'elles se découpent parfaitement. En réalité, les derniers datasets libres ne tombent pas exactement dans les intervalles restants. Un ou deux modèles pleins n'ont pas leurs données libres et passent à demi-valeur. Comme le solveur les a choisis **parce qu'**ils valaient beaucoup à pleine valeur, chacun coûte environ 500 points.

## 6. Réparer
On reprend l'idée du mouvement `exchange` du notebook 10, mais de façon **systématique** au lieu d'aléatoire. Pour chaque modèle à demi-valeur, du plus gros gain possible au plus petit :
1. on essaie de le passer à pleine valeur avec les données libres restantes ;
2. sinon, on cherche un modèle plein **moins précieux** qui partage un type, et on échange : le précieux prend les données libres, l'autre passe aux données sous droits.

On recommence tant qu'une réparation réussit. Pour pouvoir annuler un essai raté, on reprend la classe `State` du notebook 10. Les sources et leurs cibles sont **gelées** : on ne les touche pas.

In [6]:
class State:
    """Solution modifiable (modèles ordinaires), avec journal d'annulation. Sources et cibles sont gelées."""
    def __init__(self, plan):
        frozen_models = {x for pair in plan["modelMappings"] for x in pair}
        self.frozen_data = [p for p in plan["dataMappings"] if p[1] in frozen_models]
        self.frozen_sources = plan["modelMappings"]
        self.data = {}
        for d, m in plan["dataMappings"]:
            if m not in frozen_models:
                self.data.setdefault(m, []).append(d)
        used = {d for d, _ in plan["dataMappings"]}
        self.pools = {(t, c): Pool([d for d in SIZE if d not in used and TYPE[d] == t and COPY[d] == c])
                      for t in "ntic" for c in (0, 1)}
        self.full = {m: not any(COPY[d] for d in ds) for m, ds in self.data.items()}
        self.energy = sum(ENERGY[m] for m in self.data) + sum(ENERGY[m] for m in frozen_models)
        self.log = []

    def remove(self, m):
        ds, full = self.data.pop(m), self.full.pop(m)
        for d in ds:
            self.pools[(TYPE[d], COPY[d])].put(d)
        self.energy -= ENERGY[m]
        self.log.append(("remove", m, ds, full))

    def add(self, m, statuses):
        if self.energy + ENERGY[m] > energy_cap:
            return False
        got = give(self.pools, m, NEEDS[m], statuses)
        if got is None:
            return False
        self.data[m] = [d for d, _ in got]
        self.full[m] = not any(COPY[d] for d in self.data[m])
        self.energy += ENERGY[m]
        self.log.append(("add", m, None, None))
        return True

    def undo(self):
        while self.log:
            op, m, ds, full = self.log.pop()
            if op == "add":
                for d in self.data.pop(m):
                    self.pools[(TYPE[d], COPY[d])].put(d)
                self.full.pop(m); self.energy -= ENERGY[m]
            else:
                for d in ds:
                    self.pools[(TYPE[d], COPY[d])].remove(d)
                self.data[m], self.full[m] = ds, full
                self.energy += ENERGY[m]

    def plan(self):
        mappings = [[int(d), int(m)] for m, ds in self.data.items() for d in ds] + self.frozen_data
        return {"dataMappings": mappings, "modelMappings": self.frozen_sources}

TYPES_OF = {m: {t for t, _, _ in needs} for m, needs in NEEDS.items()}
gain_full = lambda m: VALUE[m] - VALUE[m] // 2            # ce que rapporte le passage à pleine valeur

def repair(plan):
    s = State(plan)
    n_repairs, improved = 0, True
    while improved:
        improved = False
        for b in sorted([m for m in s.data if not s.full[m]], key=gain_full, reverse=True):
            if b not in s.data or s.full[b]:
                continue
            s.log = []; s.remove(b)                       # 1. passage direct à pleine valeur
            if s.add(b, (0,)):
                n_repairs += 1; improved = True; continue
            s.undo()
            partners = sorted([a for a in s.data if s.full[a] and TYPES_OF[a] & TYPES_OF[b] and gain_full(a) < gain_full(b)],
                              key=gain_full)
            for a in partners[:40]:                       # 2. échange avec un modèle plein moins précieux
                s.log = []; s.remove(a); s.remove(b)
                if s.add(b, (0,)) and s.add(a, (1, 0)):
                    n_repairs += 1; improved = True; break
                s.undo()
    return s.plan(), n_repairs

t0 = time.time()
plan_h, n = repair(plan_g)
print(f"{n} réparations en {time.time() - t0:.0f} s")
evaluate("H : G + réparation", plan_h)

5 réparations en 1 s
H : G + réparation : score = 272,207 | valide = True  | écart au gagnant = -115


272207

## 7. Boucler : interdire les modèles qui ne se casent pas
Les modèles ratés à la partie 5 posent problème **à cause du découpage des datasets**, que le solveur ne voit pas. On le lui dit : on interdit à ces modèles la pleine valeur ($x_m = 0$) et on relance le solveur. Il trouve une autre sélection, presque aussi bonne sur le papier, mais plus facile à découper.

C'est une boucle classique en recherche opérationnelle : **le modèle propose, la réalité corrige, le modèle apprend**.

In [7]:
sol_i, info_i = solve_model("SCIP", seconds=120, forbid_full=set(failed))
display(pd.DataFrame([info_i]).set_index("solveur"))
plan_i0, failed_i = pack(sol_i)
print("modèles pleins ratés :", failed_i)
plan_i, n = repair(plan_i0)
print(f"{n} réparations")
evaluate("I : boucle + réparation", plan_i)

,objectif,borne,écart_prouvé,secondes,pleins,demi,impossibles,sources
solveur,,,,,,,,
SCIP,272308,272384,76,121,537,328,8,9


modèles pleins ratés : [2685]


3 réparations
I : boucle + réparation : score = 272,224 | valide = True  | écart au gagnant = -98


272224

## 8. Comparaison

In [8]:
summary = pd.DataFrame(results).T
summary.loc["E : glouton + PL (notebook 09)", ["score", "écart_au_gagnant"]] = [score_e, score_e - WINNER]
summary.loc["F : E + recuit (notebook 10)", ["score", "écart_au_gagnant"]] = [271_874, 271_874 - WINNER]
summary.loc["gagnant du concours", ["score", "écart_au_gagnant"]] = [WINNER, 0]
summary.loc["borne du solveur", ["score", "écart_au_gagnant"]] = [info_scip["borne"], info_scip["borne"] - WINNER]
summary.sort_values("score")

,score,valide,écart_au_gagnant,part_du_gagnant
G : solveur + datasets,271315,True,-1007,99.630%
E : glouton + PL (notebook 09),271511,NaN,-811,NaN
F : E + recuit (notebook 10),271874,NaN,-448,NaN
H : G + réparation,272207,True,-115,99.958%
I : boucle + réparation,272224,True,-98,99.964%
gagnant du concours,272322,NaN,0,NaN
borne du solveur,272396,NaN,74,NaN


**Lecture**
- Le solveur **seul** (G) fait moins bien que le recuit : l'agrégation est trop optimiste sur les données libres, et un ou deux modèles précieux ne se casent pas.
- La **réparation** (H) récupère l'essentiel : on dépasse nettement le plan F.
- La **boucle** (I) donne au solveur l'information qui lui manquait. On arrive à environ 100 points du gagnant, soit **99,96 %** de son score.
- Il reste au plus 74 points entre la borne et le gagnant. Le gagnant était donc très près de l'optimum.

In [9]:
best_name = max(results, key=lambda k: results[k]["score"])
best_plan = {"G : solveur + datasets": plan_g, "H : G + réparation": plan_h, "I : boucle + réparation": plan_i}[best_name]
score, valid, message = getSolutionScore(json.dumps(best_plan), DATASET_TXT, show_details=True)
print("meilleur :", best_name, "| score officiel =", score, "| valide =", valid, message)


Total value: 272224
Energy used: 209398363954637 / 209400189201537 (100.00%)
Number of used models: 882 / 3000 (29.40%)
  Number of devalued models: 344 (39.00%)
  Number of models used as data source: 9 (1.02%)
Number of used datasets: 5184 / 10000 (51.84%)
meilleur : I : boucle + réparation | score officiel = 272224 | valide = True 


## 9. Écrire le fichier de solution
On ne remplace la solution principale que si on l'améliore.

In [10]:
out_dir = ROOT_DIR / "solutions"
(out_dir / "2_medium_ortools.json").write_text(json.dumps(best_plan))
main = out_dir / "2_medium_main.json"
current = getSolutionScore(main.read_text(), DATASET_TXT, show_details=False)[0] if main.exists() else 0
if valid and score > current:
    main.write_text(json.dumps(best_plan))
    print(f"2_medium_main.json mis à jour : {current:,} -> {score:,}")
else:
    print(f"2_medium_main.json inchangé ({current:,})")

2_medium_main.json mis à jour : 271,874 -> 272,224


## À retenir
1. **Modéliser, c'est choisir ce qu'on oublie.** Le modèle agrégé oublie le découpage en datasets. C'est ce qui le rend soluble, et c'est aussi la source de son erreur.
2. **Un solveur donne deux nombres** : une solution et une borne. La borne dit combien il reste au maximum à gagner. Elle nous a aussi permis de repérer une **hypothèse fausse** : le score du gagnant dépassait la borne du notebook 10.
3. **Les sources valaient le coup, mais pas comme je les cherchais.** Le solveur a trouvé l'arbitrage en considérant toutes les combinaisons. Une règle gloutonne ne voit qu'un choix à la fois.
4. **L'exact et l'heuristique se complètent encore.** Solveur pour la sélection, `fill` pour les datasets, réparation pour corriger l'écart entre le modèle et la réalité, puis une boucle pour que le modèle tienne compte de cet écart.
5. **Le choix du solveur dépend de la structure du problème.** Un problème presque linéaire, avec de gros coefficients, favorise un solveur MIP comme SCIP. CP-SAT brille plutôt sur les problèmes très combinatoires : ordonnancement, affectations, contraintes logiques.